## **1. Environment Setup**

In [1]:
import os
import warnings
from transformers import logging as hf_logging
 
warnings.filterwarnings("ignore", message=".*kernel version.*")
warnings.filterwarnings("ignore", category=UserWarning)
hf_logging.set_verbosity_error()
 
print("Environment secured. Safe to import heavy libraries.")

Environment secured. Safe to import heavy libraries.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
 
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

In [3]:
if torch.cuda.is_available():
    print(f"GPU Active: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
else:
    print("WARNING: NO GPU DETECTED! Ensure you are not on the login node.")

GPU Active: NVIDIA A30 (25.3 GB VRAM)


In [4]:
LABEL_NAMES = [
    "Critical Rescue",            # 0
    "Resource Requests",          # 1
    "Situational Awareness",      # 2
    "Volunteering and Donations", # 3
    "Irrelevant",                 # 4
]
label2id = {label: i for i, label in enumerate(LABEL_NAMES)}
id2label  = {i: label for label, i in label2id.items()}

In [5]:
MODELS_REGISTRY = {
    "local_mbert":       "bert-base-multilingual-cased",
    "local_xlm_roberta": "xlm-roberta-base",
    "local_muril":       "google/muril-base-cased",
    "local_indic_bert":  "ai4bharat/indic-bert",
}
 
# ── Hyperparameters ────────────────────────────────────────────────────────────
LEARNING_RATE    = 2e-5
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE  = 64
GRAD_ACCUM_STEPS = 1
EPOCHS           = 3
MAX_SEQ_LENGTH   = 128
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
RANDOM_SEED      = 42

In [6]:
current_dir = Path.cwd()
BASE_DIR    = current_dir.parent if current_dir.name in ("src", "notebooks") else current_dir
 
REAL_CSV           = BASE_DIR / "datasets" / "processed" / "humaid_processed.csv"
AUGMENTED_CSV      = BASE_DIR / "datasets" / "processed" / "humaid_train_augmented.csv"
OFFLINE_MODELS_DIR = BASE_DIR / "offline_models"
 
# Weights go to trained_models/{model_key}_{suffix}/  (no timestamp)
TRAINED_MODELS_DIR = BASE_DIR / "trained_models"
 
# Training artefacts (logs, CSVs, checkpoints) go under results/Training/
RESULTS_DIR = BASE_DIR / "results" / "Training"
 
TRAINED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
 
print(f"Base directory  : {BASE_DIR}")
print(f"Trained models  : {TRAINED_MODELS_DIR}")
print(f"Training results: {RESULTS_DIR}")

Base directory  : /home/aakash/rizwan/NLP_Project
Trained models  : /home/aakash/rizwan/NLP_Project/trained_models
Training results: /home/aakash/rizwan/NLP_Project/results/Training


In [7]:
def tokenize_dataset(hf_dataset, tokenizer):
    def _tok(batch):
        return tokenizer(
            batch["clean_text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        )
    return hf_dataset.map(_tok, batched=True)

In [8]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
 
    acc = accuracy_score(labels, preds)
    macro_p,  macro_r,  macro_f1,  _ = precision_recall_fscore_support(
        labels, preds, average="macro",    zero_division=0)
    weight_p, weight_r, weight_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0)
    class_p,  class_r,  class_f1,  _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0,
        labels=list(range(len(LABEL_NAMES))))
 
    metrics = {
        "accuracy":           acc,
        "macro_precision":    macro_p,  "macro_recall":    macro_r,  "macro_f1":    macro_f1,
        "weighted_precision": weight_p, "weighted_recall": weight_r, "weighted_f1": weight_f1,
    }
    for i, name in enumerate(LABEL_NAMES):
        safe = name.replace(" ", "_").lower()
        metrics[f"precision_class_{safe}"] = float(class_p[i])
        metrics[f"recall_class_{safe}"]    = float(class_r[i])
        metrics[f"f1_class_{safe}"]        = float(class_f1[i])
    return metrics

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma
 
    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(inputs, targets, weight=self.weight, reduction="none")
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [10]:
def make_weighted_trainer(weights_tensor):
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fn = FocalLoss(weight=weights_tensor, gamma=2.0)
            loss    = loss_fn(
                outputs.logits.view(-1, self.model.config.num_labels),
                labels.view(-1),
            )
            return (loss, outputs) if return_outputs else loss
    return WeightedTrainer

In [11]:
def load_splits(use_augmentation: bool):
    if not REAL_CSV.exists() or (use_augmentation and not AUGMENTED_CSV.exists()):
        raise FileNotFoundError("Missing CSV files. Please check dataset paths.")
 
    real_df = pd.read_csv(REAL_CSV)
    dev_df  = real_df[real_df["split"] == "dev"].copy()
    test_df = real_df[real_df["split"] == "test"].copy()
 
    if use_augmentation:
        train_df = pd.read_csv(AUGMENTED_CSV)
        print("Using AUGMENTED training dataset.")
    else:
        train_df = real_df[real_df["split"] == "train"].copy()
        print("Using ORIGINAL (No Augmentation) training dataset.")
 
    for df in [train_df, dev_df, test_df]:
        df["clean_text"] = df["clean_text"].astype(str).fillna("")
        df["label"]      = df["target_label"].map(label2id).astype(int)
 
    return train_df, dev_df, test_df

In [12]:
def train_experiment(model_name: str, use_augmentation: bool):
    if model_name not in MODELS_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}")
 
    train_df, dev_df, test_df = load_splits(use_augmentation)
    suffix = "aug" if use_augmentation else "no_aug"
 
    model_path = OFFLINE_MODELS_DIR / model_name
 
    # ── Output directories (no timestamp) ─────────────────────────────
    # Weights:   trained_models/{model_name}_{suffix}/
    # Artefacts: results/Training/{model_name}_{suffix}/
    run_dir         = RESULTS_DIR        / f"{model_name}_{suffix}"
    final_model_dir = TRAINED_MODELS_DIR / f"{model_name}_{suffix}"
    run_dir.mkdir(parents=True, exist_ok=True)
    final_model_dir.mkdir(parents=True, exist_ok=True)
 
    print("=" * 70)
    print(f"TRAINING: {model_name}  |  AUGMENTATION: {use_augmentation}")
    print(f"Train Size: {len(train_df)}  |  Dev Size: {len(dev_df)}")
    print(f"Weights  → {final_model_dir}")
    print(f"Results  → {run_dir}")
    print("=" * 70)
 
    # ── Class weights ──────────────────────────────────────────────────
    device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    class_weights = compute_class_weight(
        "balanced",
        classes=np.arange(len(LABEL_NAMES)),
        y=train_df["label"].values,
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
 
    # ── Model & Tokenizer ──────────────────────────────────────────────
    use_fast  = "indic-bert" not in str(model_path).lower()
    tokenizer = AutoTokenizer.from_pretrained(
        str(model_path), use_fast=use_fast, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        str(model_path),
        num_labels=len(LABEL_NAMES),
        id2label=id2label,
        label2id=label2id,
        local_files_only=True,
    )
 
    # ── Tokenize ───────────────────────────────────────────────────────
    cols_needed = ["clean_text", "label"]
    train_hf = tokenize_dataset(
        Dataset.from_pandas(train_df[cols_needed].reset_index(drop=True)), tokenizer
    ).rename_column("label", "labels")
    dev_hf = tokenize_dataset(
        Dataset.from_pandas(dev_df[cols_needed].reset_index(drop=True)), tokenizer
    ).rename_column("label", "labels")
 
    fmt_cols = ["input_ids", "attention_mask", "labels"]
    if "token_type_ids" in train_hf.column_names:
        fmt_cols.append("token_type_ids")
    for ds in [train_hf, dev_hf]:
        ds.set_format("torch", columns=fmt_cols)
 
    # ── Training Arguments ─────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=str(run_dir / "checkpoints"),
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        logging_dir=str(run_dir / "logs"),
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        optim="adamw_torch_fused",
        bf16=True,
        tf32=True,
        fp16=False,
        report_to="none",
        dataloader_num_workers=0,
        seed=RANDOM_SEED,
    )
 
    # ── Train ──────────────────────────────────────────────────────────
    WeightedTrainer = make_weighted_trainer(weights_tensor)
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_hf,
        eval_dataset=dev_hf,
        compute_metrics=compute_metrics,
    )
    trainer.train()
 
    # ── Save model weights ─────────────────────────────────────────────
    # Saved directly to trained_models/{model_name}_{suffix}/
    trainer.save_model(str(final_model_dir))
    tokenizer.save_pretrained(str(final_model_dir))
    print(f"\nModel weights saved → {final_model_dir}")
 
    # ── Evaluate on dev set ────────────────────────────────────────────
    print("\nEvaluating on Dev Set...")
    dev_results = trainer.evaluate(dev_hf)
 
    log_history = trainer.state.log_history
    train_loss  = next(
        (log["loss"] for log in reversed(log_history) if "loss" in log), None
    )
 
    final_metrics = {
        "model_name":       model_name,
        "augmentation":     suffix,
        "train_loss":       train_loss,
        "val_loss":         dev_results.get("eval_loss"),
    }
    for k, v in dev_results.items():
        if k.startswith("eval_") and k != "eval_loss":
            final_metrics[k.replace("eval_", "")] = v
 
    print(f"  Training Loss : {final_metrics['train_loss']:.4f}")
    print(f"  Validation Loss: {final_metrics['val_loss']:.4f}")
    print(f"  Accuracy      : {final_metrics['accuracy']:.4f}")
    print(f"  Macro F1      : {final_metrics['macro_f1']:.4f}")
 
    # ── Save metrics CSV ───────────────────────────────────────────────
    # Path: results/Training/{model_name}_{suffix}/model_train_results_{suffix}.csv
    csv_path = run_dir / f"model_train_results_{suffix}.csv"
    pd.DataFrame([final_metrics]).to_csv(csv_path, index=False)
    print(f"\nFinished {model_name}! Metrics saved → {csv_path}\n")

In [13]:
MODEL_TO_TRAIN = "local_muril"
 
print(">>> STARTING PRE-AUGMENTATION RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=False)
 
print("\n>>> STARTING AUGMENTED RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=True)

>>> STARTING PRE-AUGMENTATION RUN <<<
Using ORIGINAL (No Augmentation) training dataset.
TRAINING: local_muril  |  AUGMENTATION: False
Train Size: 51952  |  Dev Size: 7564
Weights  → /home/aakash/rizwan/NLP_Project/trained_models/local_muril_no_aug
Results  → /home/aakash/rizwan/NLP_Project/results/Training/local_muril_no_aug


Map:   0%|          | 0/51952 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0709, 'grad_norm': 0.16859422624111176, 'learning_rate': 4.016393442622951e-06, 'epoch': 0.06157635467980296}
{'loss': 1.0448, 'grad_norm': 0.23861391842365265, 'learning_rate': 8.114754098360657e-06, 'epoch': 0.12315270935960591}
{'loss': 0.9993, 'grad_norm': 1.6313170194625854, 'learning_rate': 1.221311475409836e-05, 'epoch': 0.18472906403940886}
{'loss': 0.9018, 'grad_norm': 2.7192301750183105, 'learning_rate': 1.6311475409836068e-05, 'epoch': 0.24630541871921183}
{'loss': 0.8642, 'grad_norm': 0.7557288408279419, 'learning_rate': 1.99543795620438e-05, 'epoch': 0.3078817733990148}
{'loss': 0.8282, 'grad_norm': 2.131901264190674, 'learning_rate': 1.9498175182481753e-05, 'epoch': 0.3694581280788177}
{'loss': 0.6802, 'grad_norm': 2.1921560764312744, 'learning_rate': 1.9041970802919708e-05, 'epoch': 0.43103448275862066}
{'loss': 0.5809, 'grad_norm': 1.722640872001648, 'learning_rate': 1.8585766423357666e-05, 'epoch': 0.49261083743842365}
{'loss': 0.5374, 'grad_norm': 3.0139553

Map:   0%|          | 0/54452 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0535, 'grad_norm': 0.07658161967992783, 'learning_rate': 3.828125000000001e-06, 'epoch': 0.05875440658049354}
{'loss': 1.0773, 'grad_norm': 0.21173296868801117, 'learning_rate': 7.734375e-06, 'epoch': 0.11750881316098707}
{'loss': 1.0292, 'grad_norm': 1.6247535943984985, 'learning_rate': 1.1640625000000002e-05, 'epoch': 0.1762632197414806}
{'loss': 0.9551, 'grad_norm': 1.3318917751312256, 'learning_rate': 1.5546875e-05, 'epoch': 0.23501762632197415}
{'loss': 0.8806, 'grad_norm': 2.932258367538452, 'learning_rate': 1.9453125e-05, 'epoch': 0.2937720329024677}
{'loss': 0.7827, 'grad_norm': 3.045926809310913, 'learning_rate': 1.962559860687854e-05, 'epoch': 0.3525264394829612}
{'loss': 0.6783, 'grad_norm': 1.5955655574798584, 'learning_rate': 1.919024814976056e-05, 'epoch': 0.4112808460634548}
{'loss': 0.5664, 'grad_norm': 12.066112518310547, 'learning_rate': 1.875489769264258e-05, 'epoch': 0.4700352526439483}
{'loss': 0.4936, 'grad_norm': 6.864349365234375, 'learning_rate': 1.8

In [14]:
MODEL_TO_TRAIN = "local_mbert"
 
print(">>> STARTING PRE-AUGMENTATION RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=False)
 
print("\n>>> STARTING AUGMENTED RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=True)

>>> STARTING PRE-AUGMENTATION RUN <<<
Using ORIGINAL (No Augmentation) training dataset.
TRAINING: local_mbert  |  AUGMENTATION: False
Train Size: 51952  |  Dev Size: 7564
Weights  → /home/aakash/rizwan/NLP_Project/trained_models/local_mbert_no_aug
Results  → /home/aakash/rizwan/NLP_Project/results/Training/local_mbert_no_aug


Map:   0%|          | 0/51952 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0759, 'grad_norm': 3.669381618499756, 'learning_rate': 4.016393442622951e-06, 'epoch': 0.06157635467980296}
{'loss': 0.9153, 'grad_norm': 6.855721950531006, 'learning_rate': 8.114754098360657e-06, 'epoch': 0.12315270935960591}
{'loss': 0.5905, 'grad_norm': 10.17076301574707, 'learning_rate': 1.221311475409836e-05, 'epoch': 0.18472906403940886}
{'loss': 0.4201, 'grad_norm': 3.1237542629241943, 'learning_rate': 1.6311475409836068e-05, 'epoch': 0.24630541871921183}
{'loss': 0.3371, 'grad_norm': 6.433817386627197, 'learning_rate': 1.99543795620438e-05, 'epoch': 0.3078817733990148}
{'loss': 0.4254, 'grad_norm': 4.3125834465026855, 'learning_rate': 1.9498175182481753e-05, 'epoch': 0.3694581280788177}
{'loss': 0.3522, 'grad_norm': 4.633349418640137, 'learning_rate': 1.9041970802919708e-05, 'epoch': 0.43103448275862066}
{'loss': 0.3069, 'grad_norm': 4.504158973693848, 'learning_rate': 1.8585766423357666e-05, 'epoch': 0.49261083743842365}
{'loss': 0.3053, 'grad_norm': 2.8524312973022

Map:   0%|          | 0/54452 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0497, 'grad_norm': 5.6208815574646, 'learning_rate': 3.828125000000001e-06, 'epoch': 0.05875440658049354}
{'loss': 0.8434, 'grad_norm': 4.224752426147461, 'learning_rate': 7.734375e-06, 'epoch': 0.11750881316098707}
{'loss': 0.4749, 'grad_norm': 9.647316932678223, 'learning_rate': 1.1640625000000002e-05, 'epoch': 0.1762632197414806}
{'loss': 0.3302, 'grad_norm': 7.8286261558532715, 'learning_rate': 1.5546875e-05, 'epoch': 0.23501762632197415}
{'loss': 0.2852, 'grad_norm': 6.634699821472168, 'learning_rate': 1.9453125e-05, 'epoch': 0.2937720329024677}
{'loss': 0.2844, 'grad_norm': 7.094645023345947, 'learning_rate': 1.962559860687854e-05, 'epoch': 0.3525264394829612}
{'loss': 0.2622, 'grad_norm': 2.379909038543701, 'learning_rate': 1.919024814976056e-05, 'epoch': 0.4112808460634548}
{'loss': 0.2493, 'grad_norm': 11.487809181213379, 'learning_rate': 1.875489769264258e-05, 'epoch': 0.4700352526439483}
{'loss': 0.229, 'grad_norm': 4.242887020111084, 'learning_rate': 1.8319547235

In [15]:
MODEL_TO_TRAIN = "local_indic_bert"
 
print(">>> STARTING PRE-AUGMENTATION RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=False)
 
print("\n>>> STARTING AUGMENTED RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=True)

>>> STARTING PRE-AUGMENTATION RUN <<<
Using ORIGINAL (No Augmentation) training dataset.
TRAINING: local_indic_bert  |  AUGMENTATION: False
Train Size: 51952  |  Dev Size: 7564
Weights  → /home/aakash/rizwan/NLP_Project/trained_models/local_indic_bert_no_aug
Results  → /home/aakash/rizwan/NLP_Project/results/Training/local_indic_bert_no_aug


Map:   0%|          | 0/51952 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0715, 'grad_norm': 0.34093543887138367, 'learning_rate': 4.016393442622951e-06, 'epoch': 0.06157635467980296}
{'loss': 1.044, 'grad_norm': 0.5194072127342224, 'learning_rate': 8.114754098360657e-06, 'epoch': 0.12315270935960591}
{'loss': 0.9556, 'grad_norm': 7.33004903793335, 'learning_rate': 1.221311475409836e-05, 'epoch': 0.18472906403940886}
{'loss': 0.715, 'grad_norm': 5.706361293792725, 'learning_rate': 1.6311475409836068e-05, 'epoch': 0.24630541871921183}
{'loss': 0.6034, 'grad_norm': 7.160861968994141, 'learning_rate': 1.99543795620438e-05, 'epoch': 0.3078817733990148}
{'loss': 0.4874, 'grad_norm': 4.804248809814453, 'learning_rate': 1.9498175182481753e-05, 'epoch': 0.3694581280788177}
{'loss': 0.4053, 'grad_norm': 4.967016220092773, 'learning_rate': 1.9041970802919708e-05, 'epoch': 0.43103448275862066}
{'loss': 0.3367, 'grad_norm': 5.627383232116699, 'learning_rate': 1.8585766423357666e-05, 'epoch': 0.49261083743842365}
{'loss': 0.342, 'grad_norm': 13.567288398742676

In [16]:
MODEL_TO_TRAIN = "local_xlm_roberta"
 
print(">>> STARTING PRE-AUGMENTATION RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=False)
 
print("\n>>> STARTING AUGMENTED RUN <<<")
train_experiment(MODEL_TO_TRAIN, use_augmentation=True)

>>> STARTING PRE-AUGMENTATION RUN <<<
Using ORIGINAL (No Augmentation) training dataset.
TRAINING: local_xlm_roberta  |  AUGMENTATION: False
Train Size: 51952  |  Dev Size: 7564
Weights  → /home/aakash/rizwan/NLP_Project/trained_models/local_xlm_roberta_no_aug
Results  → /home/aakash/rizwan/NLP_Project/results/Training/local_xlm_roberta_no_aug


Map:   0%|          | 0/51952 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0889, 'grad_norm': 3.4023547172546387, 'learning_rate': 4.016393442622951e-06, 'epoch': 0.06157635467980296}
{'loss': 1.0035, 'grad_norm': 8.5631742477417, 'learning_rate': 8.114754098360657e-06, 'epoch': 0.12315270935960591}
{'loss': 0.6535, 'grad_norm': 9.733916282653809, 'learning_rate': 1.221311475409836e-05, 'epoch': 0.18472906403940886}
{'loss': 0.4468, 'grad_norm': 5.082375526428223, 'learning_rate': 1.6311475409836068e-05, 'epoch': 0.24630541871921183}
{'loss': 0.3641, 'grad_norm': 7.012787818908691, 'learning_rate': 1.99543795620438e-05, 'epoch': 0.3078817733990148}
{'loss': 0.4119, 'grad_norm': 8.915000915527344, 'learning_rate': 1.9498175182481753e-05, 'epoch': 0.3694581280788177}
{'loss': 0.3569, 'grad_norm': 7.682788848876953, 'learning_rate': 1.9041970802919708e-05, 'epoch': 0.43103448275862066}
{'loss': 0.3186, 'grad_norm': 4.655681133270264, 'learning_rate': 1.8585766423357666e-05, 'epoch': 0.49261083743842365}
{'loss': 0.3077, 'grad_norm': 7.767462253570557,

Map:   0%|          | 0/54452 [00:00<?, ? examples/s]

Map:   0%|          | 0/7564 [00:00<?, ? examples/s]

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


{'loss': 1.0701, 'grad_norm': 2.6286511421203613, 'learning_rate': 3.828125000000001e-06, 'epoch': 0.05875440658049354}
{'loss': 1.0264, 'grad_norm': 5.652170181274414, 'learning_rate': 7.734375e-06, 'epoch': 0.11750881316098707}
{'loss': 0.5965, 'grad_norm': 10.275527954101562, 'learning_rate': 1.1640625000000002e-05, 'epoch': 0.1762632197414806}
{'loss': 0.358, 'grad_norm': 7.060506820678711, 'learning_rate': 1.5546875e-05, 'epoch': 0.23501762632197415}
{'loss': 0.29, 'grad_norm': 9.15002155303955, 'learning_rate': 1.9453125e-05, 'epoch': 0.2937720329024677}
{'loss': 0.2959, 'grad_norm': 12.328348159790039, 'learning_rate': 1.962559860687854e-05, 'epoch': 0.3525264394829612}
{'loss': 0.2614, 'grad_norm': 3.8998773097991943, 'learning_rate': 1.919024814976056e-05, 'epoch': 0.4112808460634548}
{'loss': 0.2501, 'grad_norm': 9.14953327178955, 'learning_rate': 1.875489769264258e-05, 'epoch': 0.4700352526439483}
{'loss': 0.2288, 'grad_norm': 3.9281463623046875, 'learning_rate': 1.831954723